# Kansere Karşı Bayesyen Yaklaşım (Naive Bayes) - Analiz Bulguları

Bu notebook, kanser tarama veri seti üzerinde **Bayesyen Yaklaşım (Naive Bayes)** makine öğrenmesi modelinin eğitilmesi, hiperparametre optimizasyonu ve performans analizini içerir. Lojistik regresyon ile karşılaştırılabilmesi için benzer bir analiz yapısı kullanılmıştır.

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, f1_score

In [2]:
# Verileri yükleyelim
print("Veriler yükleniyor...")
X_train = pd.read_csv('../X_train_hazir.csv')
X_test = pd.read_csv('../X_test_hazir.csv')
y_train = pd.read_csv('../y_train_hazir.csv').squeeze().astype(int)
y_test = pd.read_csv('../y_test_hazir.csv').squeeze().astype(int)

print(f"Eğitim seti: {X_train.shape}")
print(f"Test seti: {X_test.shape}")

Veriler yükleniyor...
Eğitim seti: (8000, 15)
Test seti: (2000, 15)


In [3]:
# Eğitilmiş en iyi Naive Bayes modelini yükleme
model_nb = joblib.load('../models/model_nb.pkl')
print("Model başarıyla yüklendi:", model_nb)

Model başarıyla yüklendi: GaussianNB(var_smoothing=0.001917910261624487)


In [4]:
# Tahminler üretelim ve değerlendirelim
y_pred = model_nb.predict(X_test)
y_prob = model_nb.predict_proba(X_test)[:, 1]

print("--- Naive Bayes Sınıflandırma Raporu ---")
print(classification_report(y_test, y_pred))
print(f"Test F1-Skoru: {f1_score(y_test, y_pred):.3f}")
print(f"Test ROC-AUC Skoru: {roc_auc_score(y_test, y_prob):.3f}")

--- Naive Bayes Sınıflandırma Raporu ---
              precision    recall  f1-score   support

           0       0.81      0.89      0.85      1605
           1       0.27      0.17      0.21       395

    accuracy                           0.75      2000
   macro avg       0.54      0.53      0.53      2000
weighted avg       0.71      0.75      0.72      2000

Test F1-Skoru: 0.207
Test ROC-AUC Skoru: 0.621


## Performans Görselleştirmeleri

Eğitilen Bayesian Approach modelinin performansını gösteren **Konfüzyon Matrisi** ve **ROC Eğrisi** grafikleri aşağıda çizilmiştir.

In [5]:
# Görselleştirme - Konfüzyon Matrisi ve ROC Eğrisi
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Konfüzyon Matrisi
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Kanser Yok', 'Kanser Var'], 
            yticklabels=['Kanser Yok', 'Kanser Var'],
            annot_kws={'size': 14, 'weight': 'bold'}, cbar=False)
axes[0].set_title('Naive Bayes - Konfüzyon Matrisi', fontsize=14, pad=15)
axes[0].set_ylabel('Gerçek Sınıf', fontsize=12)
axes[0].set_xlabel('Tahmin Sınıfı', fontsize=12)

# ROC Eğrisi
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Eğrisi (AUC = {roc_auc_score(y_test, y_prob):.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('Naive Bayes - ROC Eğrisi', fontsize=14, pad=15)
axes[1].legend(loc="lower right", fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Sonuç ve Yorumlar

- **Bayesyen Yaklaşım (Naive Bayes)**, Lojistik regresyon ile benzer performans hedeflerine ulaşmıştır.
- Sınıflandırma raporuna göre, **Kanser Yok (Sınıf 0)** tespiti oldukça başarılıyken, sınıf imbalanced (dengesiz dağılım) yapısı sebebiyle **Kanser Var (Sınıf 1)** hassasiyetleri daha düşüktür.
- Grid Search hiperparametre optimizasyonu ile en uygun `var_smoothing` değeri `1.92e-03` olarak belirlenmiş ve F1 skoru maksimize edilmiştir.